# Hands-On: Avaliação de LLMs e Agents

Este notebook demonstra como avaliar LLMs e Agents usando métricas baseadas em LLM-as-a-Judge com DeepEval.

## Objetivos

1. Aprender a usar métricas genéricas de LLM (Answer Relevancy, Faithfulness, Bias/Toxicity)
2. Aprender a usar métricas específicas de Agents (Task Completion, Tool Correctness, etc.)
3. Praticar avaliação em batch
4. Gerar relatórios e visualizações


In [ ]:
# Setup e Instalação
import sys
from pathlib import Path

# Adicionar src ao path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "src"))

# Verificar se as dependências estão instaladas
try:
    import deepeval
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import google.generativeai as genai
    print("✓ Todas as dependências estão instaladas")
except ImportError as e:
    print(f"✗ Erro ao importar dependências: {e}")
    print("Execute: pip install -r requirements.txt")


In [ ]:
# Configuração de API Keys (Gemini)
import os
from dotenv import load_dotenv

load_dotenv()

# Verificar se as chaves estão configuradas
gemini_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if gemini_key:
    print("✓ Chave da API do Gemini configurada")
else:
    print("⚠ GEMINI_API_KEY (ou GOOGLE_API_KEY) não encontrada. Configure no arquivo .env")


## Parte 1: Avaliação de LLM

Vamos começar avaliando respostas de LLM usando métricas genéricas.


In [ ]:
# Importar funções de avaliação de LLM
from src.metrics_llm import (
    evaluate_answer_relevancy,
    evaluate_faithfulness,
    evaluate_bias_toxicity,
    evaluate_geval_custom,
    evaluate_llm_batch
)

from src.utils import load_llm_dataset, format_results, generate_report, plot_metrics

print("✓ Funções importadas com sucesso")


In [ ]:
# Carregar dataset de teste
dataset_path = project_root / "data" / "test_dataset_llm.json"
test_cases = load_llm_dataset(str(dataset_path))

print(f"✓ Carregados {len(test_cases)} casos de teste")
print(f"\nPrimeiro caso de teste:")
print(f"Input: {test_cases[0]['input']}")
print(f"Output: {test_cases[0]['actual_output'][:100]}...")


### Exemplo 1: Answer Relevancy

Avalia se a resposta é relevante à pergunta.


In [ ]:
# Avaliar relevância de uma resposta
test_case = test_cases[0]

result = evaluate_answer_relevancy(
    input_text=test_case['input'],
    actual_output=test_case['actual_output'],
    expected_output=test_case.get('expected_output'),
    threshold=0.5
)

print(f"Métrica: {result['metric_name']}")
print(f"Score: {result['score']:.3f}")
print(f"Sucesso: {'✓' if result['success'] else '✗'}")
print(f"\nRazão: {result['reason']}")


### Exemplo 2: Faithfulness (Detecção de Alucinações)

Avalia se a resposta está baseada no contexto fornecido.


In [ ]:
# Avaliar fidelidade ao contexto
test_case = test_cases[2]  # Caso com contexto

if test_case.get('context'):
    result = evaluate_faithfulness(
        input_text=test_case['input'],
        actual_output=test_case['actual_output'],
        context=test_case['context'],
        threshold=0.5
    )
    
    print(f"Métrica: {result['metric_name']}")
    print(f"Score: {result['score']:.3f}")
    print(f"Sucesso: {'✓' if result['success'] else '✗'}")
    print(f"Tem alucinação: {'Sim' if result.get('has_hallucination') else 'Não'}")
    print(f"\nRazão: {result['reason']}")
else:
    print("Este caso de teste não possui contexto")


### Exemplo 3: Bias/Toxicity

Avalia se a resposta contém viés ou toxicidade.


In [ ]:
# Avaliar segurança da resposta
test_case = test_cases[0]

result = evaluate_bias_toxicity(
    input_text=test_case['input'],
    actual_output=test_case['actual_output'],
    threshold=0.7
)

print(f"Métrica: {result['metric_name']}")
print(f"Score: {result['score']:.3f}")
print(f"Sucesso: {'✓' if result['success'] else '✗'}")
print(f"É seguro: {'Sim' if result.get('is_safe') else 'Não'}")
print(f"\nRazão: {result['reason']}")


### Exemplo 4: G-Eval Custom

Métrica customizada baseada em critérios em linguagem natural.


In [ ]:
# Avaliar com critério customizado
test_case = test_cases[1]  # Caso com critério

if test_case.get('criteria'):
    result = evaluate_geval_custom(
        input_text=test_case['input'],
        actual_output=test_case['actual_output'],
        criteria=test_case['criteria'],
        threshold=0.5
    )
    
    print(f"Métrica: {result['metric_name']}")
    print(f"Critério: {result['criteria']}")
    print(f"Score: {result['score']:.3f}")
    print(f"Sucesso: {'✓' if result['success'] else '✗'}")
    print(f"\nRazão: {result['reason']}")
else:
    print("Este caso de teste não possui critério customizado")


### Avaliação em Batch

Avaliar múltiplos casos de teste de uma vez.


In [ ]:
# Preparar casos de teste
formatted_cases = []
for case in test_cases[:5]:  # Usar apenas os primeiros 5 para exemplo
    formatted_case = {
        'input': case.get('input', ''),
        'actual_output': case.get('actual_output', ''),
        'expected_output': case.get('expected_output'),
        'context': case.get('context')
    }
    if 'criteria' in case:
        formatted_case['criteria'] = case['criteria']
    formatted_cases.append(formatted_case)

# Executar avaliação em batch
print("Executando avaliação em batch...")
results = evaluate_llm_batch(
    test_cases=formatted_cases,
    metrics=['relevancy', 'faithfulness', 'bias_toxicity'],
    thresholds={
        'relevancy': 0.5,
        'faithfulness': 0.5,
        'bias_toxicity': 0.7
    }
)

print(f"✓ Avaliação concluída para {len(results)} casos")


In [ ]:
# Visualizar resultados em DataFrame
df = format_results(results, output_format='dataframe')
print(df.to_string())


In [ ]:
# Gerar visualizações
plot_metrics(results, figsize=(14, 6))


## Parte 2: Avaliação de Agents

Agora vamos avaliar execuções de agents usando métricas específicas.


In [ ]:
# Importar funções de avaliação de agents
from src.metrics_agents import (
    AgentExecution,
    AgentAction,
    evaluate_task_completion,
    evaluate_tool_correctness,
    evaluate_argument_correctness,
    evaluate_plan_quality,
    evaluate_plan_adherence,
    evaluate_step_efficiency,
    evaluate_agent_batch
)

from src.utils import load_agent_dataset

print("✓ Funções importadas com sucesso")


In [ ]:
# Carregar dataset de agentes
agent_dataset_path = project_root / "data" / "test_dataset_agents.json"
agent_test_cases = load_agent_dataset(str(agent_dataset_path))

print(f"✓ Carregados {len(agent_test_cases)} casos de teste de agentes")
print(f"\nPrimeiro caso de teste:")
print(f"Task: {agent_test_cases[0]['task']}")
print(f"Expected Tools: {agent_test_cases[0]['expected_tools']}")


In [ ]:
# Função auxiliar para criar AgentExecution a partir do dataset
def create_agent_execution_from_dict(data: dict) -> AgentExecution:
    execution_data = data.get('execution', {})
    
    actions = []
    for action_data in execution_data.get('actions', []):
        action = AgentAction(
            tool_name=action_data['tool_name'],
            arguments=action_data['arguments'],
            result=action_data['result'],
            step_number=action_data['step_number']
        )
        actions.append(action)
    
    execution = AgentExecution(
        task=data['task'],
        plan=execution_data.get('plan', []),
        actions=actions,
        final_output=execution_data.get('final_output')
    )
    
    return execution

# Criar execução de exemplo
test_case = agent_test_cases[0]
execution = create_agent_execution_from_dict(test_case)

print(f"Task: {execution.task}")
print(f"Plan: {execution.plan}")
print(f"Actions: {len(execution.actions)}")
print(f"Final Output: {execution.final_output}")


### Exemplo 1: Task Completion

Avalia se o agente completou a tarefa com sucesso.


In [ ]:
# Avaliar conclusão da tarefa
result = evaluate_task_completion(
    agent_execution=execution,
    expected_output=test_case.get('expected_output'),
    threshold=0.7
)

print(f"Métrica: {result['metric_name']}")
print(f"Score: {result['score']:.3f}")
print(f"Sucesso: {'✓' if result['success'] else '✗'}")
print(f"\nRazão: {result['reason']}")


### Exemplo 2: Tool Correctness

Avalia se as ferramentas corretas foram chamadas.


In [ ]:
# Avaliar correção das ferramentas
result = evaluate_tool_correctness(
    agent_execution=execution,
    expected_tools=test_case.get('expected_tools'),
    threshold=0.8
)

print(f"Métrica: {result['metric_name']}")
print(f"Score: {result['score']:.3f}")
print(f"Sucesso: {'✓' if result['success'] else '✗'}")
print(f"Ferramentas Usadas: {result['used_tools']}")
print(f"Ferramentas Esperadas: {result['expected_tools']}")
print(f"\nRazão: {result['reason']}")


### Exemplo 3: Plan Quality

Avalia a qualidade do plano de ação.


In [ ]:
# Avaliar qualidade do plano
result = evaluate_plan_quality(
    agent_execution=execution,
    threshold=0.7
)

print(f"Métrica: {result['metric_name']}")
print(f"Score: {result['score']:.3f}")
print(f"Sucesso: {'✓' if result['success'] else '✗'}")
print(f"\nRazão: {result['reason']}")


### Exemplo 4: Plan Adherence

Avalia se o agente seguiu seu próprio plano.


In [ ]:
# Avaliar aderência ao plano
result = evaluate_plan_adherence(
    agent_execution=execution,
    threshold=0.7
)

print(f"Métrica: {result['metric_name']}")
print(f"Score: {result['score']:.3f}")
print(f"Sucesso: {'✓' if result['success'] else '✗'}")
print(f"Passos no Plano: {result['plan_steps']}")
print(f"Ações Realizadas: {result['actions_taken']}")
print(f"\nRazão: {result['reason']}")


### Avaliação em Batch de Agents

Avaliar múltiplas execuções de agentes de uma vez.


In [ ]:
# Preparar execuções de agentes
agent_executions = []
for case in agent_test_cases[:3]:  # Usar apenas os primeiros 3 para exemplo
    try:
        execution = create_agent_execution_from_dict(case)
        agent_executions.append(execution)
    except Exception as e:
        print(f"Erro ao processar caso: {e}")

# Executar avaliação em batch
print("Executando avaliação em batch de agents...")
agent_results = evaluate_agent_batch(
    agent_executions=agent_executions,
    metrics=['task_completion', 'tool_correctness', 'plan_quality', 'plan_adherence'],
    thresholds={
        'task_completion': 0.7,
        'tool_correctness': 0.8,
        'plan_quality': 0.7,
        'plan_adherence': 0.7
    }
)

print(f"✓ Avaliação concluída para {len(agent_results)} execuções")


In [ ]:
# Visualizar resultados
df_agents = format_results(agent_results, output_format='dataframe')
print(df_agents.to_string())


In [ ]:
# Gerar visualizações para agents
plot_metrics(agent_results, figsize=(14, 6))


## Geração de Relatórios

Gerar relatórios completos dos resultados.


In [ ]:
# Gerar relatório de LLM
report_llm = generate_report(results, include_details=True)
print(report_llm)


In [ ]:
# Gerar relatório de Agents
report_agents = generate_report(agent_results, include_details=True)
print(report_agents)


## Exercícios Práticos

Agora é sua vez! Tente avaliar seus próprios casos de teste.

### Exercício 1: Criar e Avaliar um Caso de Teste de LLM

Crie um caso de teste personalizado e avalie usando as métricas aprendidas.


In [ ]:
# SEU CÓDIGO AQUI
# Crie um caso de teste personalizado
my_test_case = {
    'input': 'Sua pergunta aqui',
    'actual_output': 'Sua resposta aqui',
    'context': 'Seu contexto aqui (opcional)'
}

# Avalie usando as métricas
# result = evaluate_answer_relevancy(...)
# print(result)


### Exercício 2: Criar e Avaliar uma Execução de Agent

Crie uma execução de agent personalizada e avalie usando as métricas de agentes.


In [ ]:
# SEU CÓDIGO AQUI
# Crie uma execução de agent personalizada
my_agent_execution = AgentExecution(
    task='Sua tarefa aqui',
    plan=['Passo 1', 'Passo 2'],
    actions=[
        AgentAction(
            tool_name='ferramenta_exemplo',
            arguments={'arg1': 'valor1'},
            result='resultado',
            step_number=1
        )
    ],
    final_output='Saída final'
)

# Avalie usando as métricas de agentes
# result = evaluate_task_completion(my_agent_execution)
# print(result)


## Conclusão

Neste hands-on você aprendeu:

1. ✅ Como usar métricas genéricas de LLM (Answer Relevancy, Faithfulness, Bias/Toxicity, G-Eval)
2. ✅ Como usar métricas específicas de Agents (Task Completion, Tool Correctness, Plan Quality, etc.)
3. ✅ Como avaliar múltiplos casos em batch
4. ✅ Como gerar relatórios e visualizações

### Próximos Passos

- Experimente com diferentes thresholds
- Crie suas próprias métricas customizadas
- Integre em pipelines de CI/CD
- Combine avaliação automática com avaliação humana

### Referências

- [DeepEval Documentation](https://docs.confident-ai.com/)
- [Confident AI - LLM Evaluation Guide](https://www.confident-ai.com/blog)
- [DeepEval Agentic Metrics](https://docs.confident-ai.com/docs/metrics-agentic)
